# Week 8: The Price is Right — Setup and Run

**Cynthia Omovoiye**

1. Setup support + Chroma (built from items_lite; requires HF_TOKEN)
2. Test components
3. Launch Gradio app

---

## How to set up and run this notebook

### Prerequisites
- **Python 3.9+** with pip
- **Hugging Face account** and token (for `items_lite` dataset)
- **LiteLLM + Ollama** (open-source): Scanner and Frontier use a local LLM by default. Install: `pip install litellm` and run [Ollama](https://ollama.ai) with e.g. `ollama pull llama3.2`
- **Optional:** `PUSHOVER_USER` + `PUSHOVER_TOKEN` for push notifications

### Getting PUSHOVER_USER and PUSHOVER_TOKEN (optional)
Pushover sends deal alerts to your phone. Without these, the app only logs alerts.

1. Sign up at [pushover.net](https://pushover.net) and install the Pushover app on your device.
2. Log in at [pushover.net](https://pushover.net), then go to **Your Applications** (or the dashboard).
3. Click **Create Application/API Token** and name it (e.g. "Price is Right"). Leave the rest as default and submit.
4. Copy the **API Token** — this is your `PUSHOVER_TOKEN`.
5. On the same site, your **User Key** (on the main dashboard) is your `PUSHOVER_USER`.
6. Add both to `.env`: `PUSHOVER_USER=your_user_key` and `PUSHOVER_TOKEN=your_api_token`.

### Running this notebook
- **Kernel:** Choose the Python interpreter that has the dependencies above.
- **Run in order:** Execute cells from top to bottom. Run the **Environment** and **Setup Chroma** sections first so support data and the vector DB are built.
- **CLI alternative:** From this folder you can also run:
  ```bash
  python3 price_is_right_app.py --setup-chroma    # one-time
  python3 price_is_right_app.py --test specialist
  python3 price_is_right_app.py --gradio          # launch UI
  ```

---

## 1. Environment

Ensure `.env` has: `HF_TOKEN` (required for items_lite), and optionally `PUSHOVER_USER`/`PUSHOVER_TOKEN`.

**LLMs:** Scanner and Frontier use **LiteLLM** with **Ollama** by default (no paid API). Set `LITELLM_MODEL=ollama/llama3.2` or run `ollama pull llama3.2`. Pricing uses our **Week 7 hybrid (Specialist)** — no LLM. Use `SCANNER_USE_MOCK=1` to skip RSS and LLM for quick tests.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

# Optional: use mock scanner (no RSS/API) for quick testing
os.environ["SCANNER_USE_MOCK"] = "1"

print("HF_TOKEN:", "set" if os.getenv("HF_TOKEN") else "not set")
print("OPENAI_API_KEY:", "set" if os.getenv("OPENAI_API_KEY") else "not set")

## 2. Setup Chroma

Build support from items_lite and Chroma. Requires HF_TOKEN in `.env`.

In [ ]:
import sys
REPO_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd()))

from price_is_right_app import CONFIG, build_chroma_from_support, _ensure_chroma, _ensure_support

# Trigger support load (builds from items_lite if cache missing)
try:
    _ensure_support()
    print("Support data OK")
except Exception as e:
    print(f"Support: {e}")

# Build Chroma if needed
if CONFIG["SUPPORT_JSON"].exists():
    build_chroma_from_support(CONFIG["CHROMA_DB_PATH"], CONFIG["SUPPORT_JSON"])
    print("Chroma built")
else:
    print("Skip Chroma (no support)")

## 3. Test components (optional)

Run these to verify each part works. You can skip to **4. Launch Gradio** if setup succeeded.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from price_is_right_app import test_specialist, test_frontier, test_scanner, test_planner

test_specialist()

In [ ]:
# Optional: ensure Chroma exists, then test Frontier agent
_ensure_chroma()
test_frontier()

In [ ]:
# With SCANNER_USE_MOCK=1 this is fast; otherwise fetches RSS
test_scanner()

In [ ]:
test_planner()

## 4. Launch Gradio

In [ ]:
from price_is_right_app import launch_gradio

launch_gradio()